In [1]:
from selenium import webdriver

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver import ActionChains
import time
import warnings

driver = webdriver.Chrome()

In [ ]:
EU_COUNTRIES = [
    'BE', 'BG', 'DE','CZ', 'DK', 'EE', 'IE', 
    'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV',
    'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 
    'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

EFTA_COUNTRIES = ['IS', 'LI', 'NO', 'CH']

EU_EFTA = EU_COUNTRIES + EFTA_COUNTRIES

In [3]:
NOT_FOUND_TEXT = "The requested page could not be found."

for idx, c in enumerate(EU_EFTA): 

    url = f'https://data.ecb.europa.eu/data/datasets/MIR/MIR.M.{c}.B.A20.J.R.A.2240.EUR.O'
    print(f"--- Attempting to process country: {c} ---")
    print(f"Current URL: {url}")
    driver.get(url)

    if NOT_FOUND_TEXT in driver.page_source:
        print(f"Skipping {c}: The requested page could not be found.")
        continue

    if idx == 0:
        cookies_button_xpath = '/html[1]/body[1]/div[4]/div[1]/div[1]/div[2]/button[2]'
        hover = ActionChains(driver)

        # Click on the deny cookies button
        deny_button = driver.find_element(By.XPATH, cookies_button_xpath)
        deny_button.click()
        time.sleep(4)

    download_button_1_xpath = 'div.compare-download:nth-child(2)'

    # Click on the download button
    find_download_button_1 = driver.find_element(By.CSS_SELECTOR, download_button_1_xpath)
    find_download_button_1.click()


    csv_button_css = 'div.download-group:nth-child(2) > p:nth-child(2) > label:nth-child(1)'

    # On the popup select csv 
    time.sleep(4)
    find_csv_option = driver.find_element(By.CSS_SELECTOR,csv_button_css)
    WebDriverWait(driver, 10).until(EC.element_to_be_clickable(find_csv_option))
    find_csv_option.click()


    download_button_2_css = '.modal-submit-download'

    # On the popup click download
    find_download_button_2 = driver.find_element(By.CSS_SELECTOR, download_button_2_css)
    WebDriverWait(driver, 10).until(EC.element_to_be_clickable(find_download_button_2))
    find_download_button_2.click()
    time.sleep(5)
    print(f"Successfully processed {c}.")
    
driver.quit()

print("All countries processed. Browser closed.") 

--- Attempting to process country: BE ---
Current URL: https://data.ecb.europa.eu/data/datasets/MIR/MIR.M.BE.B.A20.J.R.A.2240.EUR.O
Successfully processed BE.
--- Attempting to process country: BG ---
Current URL: https://data.ecb.europa.eu/data/datasets/MIR/MIR.M.BG.B.A20.J.R.A.2240.EUR.O
Skipping BG: The requested page could not be found.
--- Attempting to process country: DE ---
Current URL: https://data.ecb.europa.eu/data/datasets/MIR/MIR.M.DE.B.A20.J.R.A.2240.EUR.O
Successfully processed DE.
--- Attempting to process country: EE ---
Current URL: https://data.ecb.europa.eu/data/datasets/MIR/MIR.M.EE.B.A20.J.R.A.2240.EUR.O
Successfully processed EE.
--- Attempting to process country: IE ---
Current URL: https://data.ecb.europa.eu/data/datasets/MIR/MIR.M.IE.B.A20.J.R.A.2240.EUR.O
Successfully processed IE.
--- Attempting to process country: EL ---
Current URL: https://data.ecb.europa.eu/data/datasets/MIR/MIR.M.EL.B.A20.J.R.A.2240.EUR.O
Skipping EL: The requested page could not be fou

In [45]:
# Read the files 
import pandas as pd
import glob
import os
import re

# path_ecb = 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/ECB data'

folder_path = 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/ECB data'
file_prefix = 'ECB Data Portal_'
file_extension = '.csv'

search_pattern = os.path.join(folder_path, file_prefix + '*' + file_extension)

all_files = glob.glob(search_pattern)

print(f"Found {len(all_files)} files matching the pattern.")
print(all_files)

Found 19 files matching the pattern.
['C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/ECB data\\ECB Data Portal_20251014102312.csv', 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/ECB data\\ECB Data Portal_20251014102324.csv', 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/ECB data\\ECB Data Portal_20251014102336.csv', 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/ECB data\\ECB Data Portal_20251014102347.csv', 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/ECB data\\ECB Data Portal_20251014102358.csv', 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/ECB data\\ECB Data Portal_20251014102409.csv', 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/ECB data\\ECB Data Portal_20251014102420.csv', 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meeting 18.10/ECB data\\ECB Data Portal_20251014102431.csv', 'C:/Users/ydmar/Documents/UW/Master thesis/Step 1 - meetin

In [ ]:
final_data = list()

for f in all_files:
    test = pd.read_csv(f)
    # print(test)

    ## Filter just years
    test['date'] = pd.to_datetime(test['DATE'])
    test['year'] = test['date'].dt.year
    test = test.drop(columns=['DATE','TIME PERIOD','date'])
    # test.info()

    ## Find the long name of the column (interest rate column)
    target_index = 0 
    current_columns = test.columns.tolist()
    old_name = current_columns[target_index]
    print(f"Old Name at Index {target_index}: {old_name}")

    ## Find the country name
    match = re.search(r'\(MIR\.M\.([A-Z]{2})\.B\.A20\.J\.R\.A\.2240\.EUR\.O\)', old_name)

    if match:
        country_name_regex = match.group(1).strip()
    print(f"Extracted Country Name (Regex): {country_name_regex}")
    test['geo'] = country_name_regex

    ## Rename the interest name column
    new_name = 'inter_rate'
    test.rename(columns={old_name: new_name}, inplace=True)

    ## Filter years (>=2016)
    test['year'] = pd.to_numeric(test['year'], errors='coerce')
    test_new = test[test['year'] >= 2016].copy()

    ## Calculate the mean for the one year
    test_new['inter_rt_mean'] = test_new.groupby('year')['inter_rate'].transform('mean')
    test_new = test_new.drop(columns=['inter_rate'])

    ## Keep unique values (interest rate per year and country)
    test_new = test_new.drop_duplicates()

    final_data.append(test_new)

if final_data:
    final_combined_df = pd.concat(final_data, ignore_index=True)

print(final_combined_df)



Old Name at Index 0: Bank interest rates - loans to corporations with an original maturity of over five years (outstanding amounts) - Belgium (MIR.M.BE.B.A20.J.R.A.2240.EUR.O)
Extracted Country Name (Regex): BE
Old Name at Index 0: Bank interest rates - loans to corporations with an original maturity of over five years (outstanding amounts) - Germany (MIR.M.DE.B.A20.J.R.A.2240.EUR.O)
Extracted Country Name (Regex): DE
Old Name at Index 0: Bank interest rates - loans to corporations with an original maturity of over five years (outstanding amounts) - Estonia (MIR.M.EE.B.A20.J.R.A.2240.EUR.O)
Extracted Country Name (Regex): EE
Old Name at Index 0: Bank interest rates - loans to corporations with an original maturity of over five years (outstanding amounts) - Ireland (MIR.M.IE.B.A20.J.R.A.2240.EUR.O)
Extracted Country Name (Regex): IE
Old Name at Index 0: Bank interest rates - loans to corporations with an original maturity of over five years (outstanding amounts) - Spain (MIR.M.ES.B.A20.

In [50]:
print(final_combined_df)

     year geo  inter_rt_mean
0    2016  BE       2.650833
1    2017  BE       2.320833
2    2018  BE       2.133333
3    2019  BE       2.010833
4    2020  BE       1.824167
..    ...  ..            ...
185  2021  FI       1.168333
186  2022  FI       1.452500
187  2023  FI       3.832500
188  2024  FI       4.350833
189  2025  FI       3.437500

[190 rows x 3 columns]


In [54]:
final_combined_df.to_csv('interest_rates_geo.csv', index = False)

In [52]:
available_countries = final_combined_df['geo'].unique()
print(available_countries)

['BE' 'DE' 'EE' 'IE' 'ES' 'FR' 'HR' 'IT' 'CY' 'LV' 'LT' 'LU' 'MT' 'NL'
 'AT' 'PT' 'SI' 'SK' 'FI']


Missed countries data: BG, CZ, DK, EL, HU, PL, RO, SE, LI, NO, CH